# Harvest readiness: first test on Kaggle

**What this notebook does.** It trains a small model that finds each oyster cluster in a photo, draws its outline and labels it **young** or **ready** to pick. This is the first try at the **Harvest readiness** model for the cultivation planner.

**Data.** Two public Roboflow sets with outlines (both CC BY 4.0, credit the owners when showing results):

| Set | Owner (Roboflow) | Images | Their labels | Our label |
|---|---|---|---|---|
| oyster-dntlz | abhi | 769 | Pinning, Early Growth / Mature | young / ready |
| mushroom-w7ucu | angelo neyra | 721 | NOT_READY / READY, OVERDUE | young / ready |

**The honest test.** Photos from one set are never used to train the model that is scored on them. Run A trains on abhi and is scored on angelo; run B does the opposite. This is the same idea as keeping the syldd photos held out: the score tells us how the model does on photos from a farm it has never seen.

**Before you press Run All (once):**
1. Right panel, *Session options*: Accelerator = **GPU T4 x2** (or P100), Internet = **On**.
2. *Add-ons, Secrets*: a secret called `ROBOFLOW_API_KEY` holding your Roboflow key, with the switch next to it turned on for this notebook.

Runtime: roughly 30 to 60 minutes on a Kaggle GPU. Nothing is saved to your PC; results land in the *Output* tab on the right.

In [ ]:
# 1. Install the two libraries we need (Kaggle has the rest)
!pip install -q roboflow ultralytics

In [ ]:
# 2. Settings: change these lines only if you want to
SETS = {
    # short name: (Roboflow workspace, project, {their label: our label})
    "abhi":   ("abhi-sok0u", "oyster-dntlz",
               {"pinning": "young", "early growth": "young", "mature": "ready"}),
    "angelo": ("angelo-neyra-9rg4y", "mushroom-w7ucu",
               {"not_ready": "young", "ready": "ready", "overdue": "ready"}),
}
OUR_LABELS = ["young", "ready"]   # class 0 = young, class 1 = ready
EPOCHS = 50                       # passes over the training photos
IMG_SIZE = 640                    # photos are shrunk to this many pixels wide
MODEL = "yolo11n-seg.pt"          # smallest outline model: fast, good enough for a first test
SEED = 0
WORK = "/kaggle/working"

In [ ]:
# 3. Connect to Roboflow with the key saved as a Kaggle secret
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow
rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))

## Pick the version without copies
Roboflow versions of one set are made by the same owner. Bigger versions usually contain flipped or rotated copies of the same photos ("augmentation"). Copies would make the model look better than it is, so for each set we list all versions and take the one with the **fewest images** (the originals). Check the table: the chosen version should show no augmentation.

In [ ]:
import pandas as pd

def version_table(ws, proj_name):
    proj = rf.workspace(ws).project(proj_name)
    rows = []
    for v in proj.versions():
        n = getattr(v, "images", None)
        if not n:
            splits = getattr(v, "splits", None) or {}
            n = sum(int(x) for x in splits.values()) if splits else None
        rows.append({"version": str(v.version).split("/")[-1],
                     "images": n,
                     "augmentation": str(getattr(v, "augmentation", ""))[:80],
                     "created": getattr(v, "created", "")})
    return proj, pd.DataFrame(rows)

CHOSEN = {}
for short, (ws, pname, _) in SETS.items():
    proj, tab = version_table(ws, pname)
    print(f"\n=== {short}: {ws}/{pname}")
    display(tab)
    tab = tab.dropna(subset=["images"])
    best = tab.sort_values(["images", "version"]).iloc[0]["version"]
    CHOSEN[short] = (proj, int(best))
    print("chosen version:", best)

In [ ]:
# 4. Download each chosen version as outlines (YOLO format)
import os
DL = {}
for short, (proj, ver) in CHOSEN.items():
    ds = proj.version(ver).download("yolov11", location=f"{WORK}/raw_{short}", overwrite=True)
    DL[short] = ds.location
    print(short, "->", ds.location)

## Translate the labels to young / ready
Each set uses its own label names. We rewrite every label file so that 0 = young and 1 = ready, and we drop any label not in the table above (printed so you can see what was dropped). All photos of one set are pooled into one folder, because the test photos come from the *other* set anyway.

In [ ]:
import glob, shutil, random, yaml, collections

def remap_set(short):
    src = DL[short]
    their = yaml.safe_load(open(f"{src}/data.yaml"))["names"]
    if isinstance(their, dict):
        their = [their[k] for k in sorted(their)]
    mapping = SETS[short][2]
    new_id = {}
    for i, name in enumerate(their):
        ours = mapping.get(str(name).strip().lower())
        if ours is not None:
            new_id[i] = OUR_LABELS.index(ours)
    print(f"{short}: their labels {their}")
    print(f"   kept as: { {their[i]: OUR_LABELS[j] for i, j in new_id.items()} }")
    out = f"{WORK}/pool_{short}"
    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(f"{out}/images"); os.makedirs(f"{out}/labels")
    counts, dropped, n_img = collections.Counter(), collections.Counter(), 0
    for split in ["train", "valid", "test"]:
        for img in glob.glob(f"{src}/{split}/images/*"):
            base = os.path.splitext(os.path.basename(img))[0]
            lab = f"{src}/{split}/labels/{base}.txt"
            lines = []
            if os.path.exists(lab):
                for line in open(lab):
                    parts = line.split()
                    if not parts:
                        continue
                    c = int(parts[0])
                    if c in new_id:
                        lines.append(" ".join([str(new_id[c])] + parts[1:]))
                        counts[OUR_LABELS[new_id[c]]] += 1
                    else:
                        dropped[their[c]] += 1
            shutil.copy(img, f"{out}/images/{split}_{os.path.basename(img)}")
            open(f"{out}/labels/{split}_{base}.txt", "w").write("\n".join(lines))
            n_img += 1
    print(f"   {n_img} photos, clusters: {dict(counts)}, dropped: {dict(dropped)}")
    return out

POOL = {s: remap_set(s) for s in SETS}

In [ ]:
# 5. Write the small settings files YOLO needs for each run.
#    Training set: 85% of its photos to learn from, 15% to check progress (same set).
#    Test set: ALL photos of the other set.
def make_run(train_set, test_set):
    random.seed(SEED)
    imgs = sorted(glob.glob(f"{POOL[train_set]}/images/*"))
    random.shuffle(imgs)
    cut = int(0.85 * len(imgs))
    run = f"{WORK}/run_{train_set}_to_{test_set}"
    os.makedirs(run, exist_ok=True)
    open(f"{run}/train.txt", "w").write("\n".join(imgs[:cut]))
    open(f"{run}/val.txt", "w").write("\n".join(imgs[cut:]))
    open(f"{run}/test.txt", "w").write("\n".join(sorted(glob.glob(f"{POOL[test_set]}/images/*"))))
    cfg = {"path": run, "train": f"{run}/train.txt", "val": f"{run}/val.txt",
           "test": f"{run}/test.txt", "names": {0: "young", 1: "ready"}}
    yaml.safe_dump(cfg, open(f"{run}/data.yaml", "w"))
    return run

RUNS = {"A: train abhi, test angelo": make_run("abhi", "angelo"),
        "B: train angelo, test abhi": make_run("angelo", "abhi")}

## Train and score
For each run: train on one set, then score on the other. The score is **mAP50 (mask)**: roughly, the share of clusters the model outlines in the right place *and* with the right label, from 0 (useless) to 1 (perfect). As a rough guide, above 0.5 on an unseen farm is a usable first model; below 0.3 means the two farms look too different and we need more varied photos.

In [ ]:
from ultralytics import YOLO
results = []
for name, run in RUNS.items():
    print("\n#####", name)
    model = YOLO(MODEL)
    model.train(data=f"{run}/data.yaml", epochs=EPOCHS, imgsz=IMG_SIZE, seed=SEED,
                project=run, name="train", exist_ok=True, verbose=False, plots=True)
    best = YOLO(f"{run}/train/weights/best.pt")
    own = best.val(data=f"{run}/data.yaml", split="val", project=run, name="val_own", exist_ok=True)
    other = best.val(data=f"{run}/data.yaml", split="test", project=run, name="test_other", exist_ok=True)
    for where, r in [("same set (held-back 15%)", own), ("OTHER set (honest score)", other)]:
        row = {"run": name, "scored on": where,
               "mAP50 mask, all": round(r.seg.map50, 3)}
        for i, lab in enumerate(OUR_LABELS):
            try:
                row[f"mAP50 mask, {lab}"] = round(r.seg.class_result(i)[2], 3)
            except Exception:
                row[f"mAP50 mask, {lab}"] = None
        results.append(row)

summary = pd.DataFrame(results)
summary.to_csv(f"{WORK}/harvest_readiness_scores.csv", index=False)
summary

In [ ]:
# Keep each run's per-epoch history for plotting: one CSV per run in history/, plus the test score, zipped
import datetime, os, re, shutil, yaml
HIST = f"{WORK}/history"
os.makedirs(HIST, exist_ok=True)
TODAY = datetime.date.today()

def save_history(model, run, direction, run_dir, train_set, test_set):
    """Copy run_dir/results.csv (one row per epoch) with the settings in front; returns the file name."""
    df = pd.read_csv(f"{run_dir}/results.csv")
    df.columns = [c.strip() for c in df.columns]
    a = yaml.safe_load(open(f"{run_dir}/args.yaml"))
    front = dict(model=model, run=run, train_set=train_set, test_set=test_set, date=TODAY.isoformat(),
                 epochs=a["epochs"], imgsz=a["imgsz"], batch=a["batch"], base_model=os.path.basename(str(a["model"])))
    for i, (k, v) in enumerate(front.items()):
        df.insert(i, k, v)
    name = "_".join(re.sub(r"[^\w-]+", "", p) for p in (model, run, direction, TODAY.strftime("%Y%m%d"))) + ".csv"
    df.to_csv(f"{HIST}/{name}", index=False)
    return name

def save_scores(model, run, table):
    """The final test score table, next to the histories."""
    t = pd.DataFrame(table)
    for i, (k, v) in enumerate({"model": model, "run": run, "date": TODAY.isoformat()}.items()):
        t.insert(i, k, v)
    t.to_csv(f"{HIST}/{model}_{run}_scores_{TODAY.strftime('%Y%m%d')}.csv", index=False)

for name, run in RUNS.items():
    tr, te = re.match(r".*run_(.+)_to_(.+)", run).groups()
    print(save_history("harvest_readiness", "run1", f"train{tr}_test{te}", f"{run}/train", tr, te))
save_scores("harvest_readiness", "run1", results)
print(shutil.make_archive(f"{WORK}/history", "zip", HIST))   # Output tab: history.zip

## Read the result
* **Same set vs OTHER set.** The same-set score is always higher, because those photos look like the training photos (often frames from the same video). The OTHER-set score is the one to trust.
* **young vs ready.** If one label scores much lower, the two farms probably draw the line between young and ready differently.

In [ ]:
# 6. Look at 6 photos from the OTHER set with the model's outlines drawn on
import matplotlib.pyplot as plt
for name, run in RUNS.items():
    best = YOLO(f"{run}/train/weights/best.pt")
    test_imgs = open(f"{run}/test.txt").read().split()
    random.seed(1)
    pick = random.sample(test_imgs, min(6, len(test_imgs)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, img in zip(axes.flat, pick):
        ax.imshow(best.predict(img, verbose=False)[0].plot()[:, :, ::-1])
        ax.axis("off")
    fig.suptitle(name + "  (green/blue outlines = model's guess)")
    plt.show()

In [ ]:
# 7. Keep the two trained models and the score table (download from the Output tab)
for name, run in RUNS.items():
    tag = run.split("run_")[-1]
    shutil.copy(f"{run}/train/weights/best.pt", f"{WORK}/harvest_readiness_{tag}.pt")
# remove the big photo folders so the Output tab stays small
for d in glob.glob(f"{WORK}/raw_*") + glob.glob(f"{WORK}/pool_*"):
    shutil.rmtree(d, ignore_errors=True)
print(os.listdir(WORK))

**Next step (separate notebook):** add Duman 2024 (Annotated Oyster Mushroom Images, boxes only, mature/immature) as a third farm, uploaded to Kaggle as a private dataset, and score a model trained on two farms on the third.